In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Labs"
)

xpt_file = DATA_DIR / "HDL_F.xpt"
csv_file = DATA_DIR / "HDL_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 HDL_F XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBDHDD",
    "LBDHDDSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HDL_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 8591:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,591 rows x 3 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HDL_F. Do not automatically replace."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifact exists."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# LBDHDDSI is a genuine continuous decimal variable.
#
# LBDHDD is whole-number mg/dL in this released file.
#
# Never round genuine decimal variables.
#

print("\n--- GENUINE DECIMAL CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


# ============================================================
# 6. VALIDATE EXPECTED DECIMAL VARIABLES
# ============================================================

DECIMAL_COLUMNS = [
    "LBDHDDSI"
]


if fractional_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs "
        "from validated HDL_F pattern.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "\nPASS: LBDHDDSI correctly identified "
    "as the genuine decimal variable."
)


print(
    "PASS: LBDHDDSI will NOT be rounded "
    "or converted to Int64."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. FORMAT WHOLE-NUMBER VARIABLES WITHOUT .0
# ============================================================

#
# LBDHDD contains only whole-number mg/dL values
# in the released HDL_F file.
#
# No .round() is used.
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBDHDD"
]


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and LBDHDD formatted "
    "without unnecessary .0."
)


print(
    "PASS: LBDHDDSI decimal values remain unchanged."
)


# ============================================================
# 9. CDC CHECK - LBDHDD
# ============================================================

#
# CDC:
#
# Direct HDL-Cholesterol (mg/dL)
#
# Range:
#     11 to 179
#
# Non-missing:
#     7,846
#
# Missing:
#     745
#

s = df["LBDHDD"]


print("\n--- LBDHDD ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 11
    and
    s.max() == 179
    and
    int(
        s.notna()
        .sum()
    )
    == 7846
    and
    int(
        s.isna()
        .sum()
    )
    == 745
):

    raise ValueError(
        "LBDHDD differs from CDC."
    )


print(
    "PASS: LBDHDD matches CDC."
)


# ============================================================
# 10. CDC CHECK - LBDHDDSI
# ============================================================

#
# CDC:
#
# Direct HDL-Cholesterol (mmol/L)
#
# Range:
#     0.28 to 4.63
#
# Non-missing:
#     7,846
#
# Missing:
#     745
#
# This is a genuine decimal variable.
#

s = df["LBDHDDSI"]


print("\n--- LBDHDDSI ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0.28,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        4.63,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 7846
    and
    int(
        s.isna()
        .sum()
    )
    == 745
):

    raise ValueError(
        "LBDHDDSI differs from CDC."
    )


print(
    "PASS: LBDHDDSI matches CDC."
)


print(
    "PASS: LBDHDDSI decimals preserved."
)


# ============================================================
# 11. HDL CONVERSION CONSISTENCY CHECK
# ============================================================

#
# CDC documentation states:
#
# LBDHDDSI = LBDHDD * 0.02586
#
# Released LBDHDDSI values are consistent with
# that conversion rounded to 2 decimal places.
#
# IMPORTANT:
#
# This check does NOT modify LBDHDDSI.
#

print("\n--- HDL CONVERSION CHECK ---")


mask = (
    df["LBDHDD"].notna()
    &
    df["LBDHDDSI"].notna()
)


expected_si = (
    df.loc[
        mask,
        "LBDHDD"
    ]
    .astype(float)
    * 0.02586
).round(2)


actual_si = (
    df.loc[
        mask,
        "LBDHDDSI"
    ]
)


if not np.allclose(
    expected_si,
    actual_si,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBDHDDSI conversion consistency check failed."
    )


print(
    "PASS: LBDHDDSI is consistent with "
    "LBDHDD x 0.02586."
)


# ============================================================
# 12. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# No tiny-zero restoration or other numeric correction
# is required for HDL_F.
#
# Therefore ALL numeric values must remain exactly
# equal to the source XPT.
#
# Only formatting changes:
#
# SEQN:
#     51624.0 -> 51624
#
# LBDHDD:
#     50.0 -> 50
#
# LBDHDDSI stays decimal.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    if col in DECIMAL_COLUMNS:

        current_values = (
            df[col]
            .to_numpy(
                dtype=float
            )
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 3 variables preserve "
    "original XPT numeric values exactly."
)


# ============================================================
# 13. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# LBDHDDSI remains decimal.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 14. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 15. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV columns match "
    "source XPT numeric values exactly."
)


# ============================================================
# 16. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 17. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifact found "
        "in final CSV."
    )


print(
    "PASS: No tiny-value artifacts."
)


# ============================================================
# 18. WHOLE-NUMBER .0 FORMAT CHECK
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally search the entire CSV for ".0".
#
# LBDHDDSI is a genuine decimal variable.
#
# Check only:
#
#     SEQN
#     LBDHDD
#

print("\n--- WHOLE-NUMBER FORMAT CHECK ---")


dot_zero_errors = []


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(
        f
    )


    for row_number, row in enumerate(
        reader,
        start=2
    ):

        for col in INTEGER_COLUMNS:

            value = row[col]


            if (
                value != ""
                and
                value.endswith(".0")
            ):

                dot_zero_errors.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


                if len(
                    dot_zero_errors
                ) >= 20:

                    break


        if len(
            dot_zero_errors
        ) >= 20:

            break


print(
    "Unnecessary '.0' errors:",
    len(
        dot_zero_errors
    )
)


if dot_zero_errors:

    print(
        dot_zero_errors
    )

    raise ValueError(
        "Unexpected .0 formatting remains "
        "in whole-number fields."
    )


print(
    "PASS: SEQN and LBDHDD have "
    "no unnecessary .0."
)


# ============================================================
# 19. EXACT DECIMAL FIDELITY
# ============================================================

#
# Explicitly verify LBDHDDSI after CSV export.
#

print("\n--- LBDHDDSI DECIMAL FIDELITY ---")


expected_values = (
    source_df["LBDHDDSI"]
    .to_numpy()
)


exported_values = (
    csv_df["LBDHDDSI"]
    .to_numpy()
)


if not np.allclose(
    expected_values,
    exported_values,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBDHDDSI decimal values changed "
        "during CSV export."
    )


print(
    "PASS: LBDHDDSI decimal values "
    "preserved exactly."
)


# ============================================================
# 20. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 21. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)

print(
    "HDL_F CONVERSION COMPLETE"
)

print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Tiny-value corrections: NONE"
)


print(
    "Whole-number fields: SEQN, LBDHDD"
)


print(
    "Decimal variables preserved: LBDHDDSI"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in whole-number fields: 0 expected"
)


print(
    "LBDHDDSI decimal values changed: 0 expected"
)

NHANES 2009-2010 HDL_F XPT -> CSV
Rows: 8,591
Columns: 3
PASS: Structure = 8,591 rows x 3 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifact exists.

--- GENUINE DECIMAL CHECK ---
LBDHDDSI: 7,845 genuine fractional values

PASS: LBDHDDSI correctly identified as the genuine decimal variable.
PASS: LBDHDDSI will NOT be rounded or converted to Int64.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: SEQN and LBDHDD formatted without unnecessary .0.
PASS: LBDHDDSI decimal values remain unchanged.

--- LBDHDD ---
Minimum: 11
Maximum: 179
Non-missing: 7846
Missing: 745
PASS: LBDHDD matches CDC.

--- LBDHDDSI ---
Minimum: 0.28
Maximum: 4.63
Non-missing: 7846
Missing: 745
PASS: LBDHDDSI matches CDC.
PASS: LBDHDDSI decimals preserved.

--- HDL CONVERSION CHECK ---
PASS: LBDHDDSI is consistent with LBDHDD x 0.02586.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: All 3